In [15]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').appName('test').getOrCreate()

In [16]:
df_green = spark.read.option("recursiveFileLookup", "true").parquet('./data/pq/green/')


In [17]:
df_green.createOrReplaceTempView('green')

In [18]:
df_green_revenue = spark.sql("""
SELECT 
    -- Reveneue grouping 
    PULocationID AS revenue_zone,
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    
    SUM(total_amount) AS amount,
    count(1) as number_records
FROM
    green
where
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
order by
    1,2
""")

In [24]:
df_green_revenue.repartition(20).write.parquet('./data/report/revenue/green', mode='overwrite')

In [25]:
df_green_revenue.explain(True)

== Parsed Logical Plan ==
'Sort [1 ASC NULLS FIRST, 2 ASC NULLS FIRST], true
+- 'Aggregate [1, 2], ['PULocationID AS revenue_zone#105, 'date_trunc(hour, 'lpep_pickup_datetime) AS hour#106, 'SUM('total_amount) AS amount#107, 'count(1) AS number_records#108]
   +- 'Filter ('lpep_pickup_datetime >= 2020-01-01 00:00:00)
      +- 'UnresolvedRelation [green], [], false

== Analyzed Logical Plan ==
revenue_zone: int, hour: timestamp, amount: double, number_records: bigint
Sort [revenue_zone#105 ASC NULLS FIRST, hour#106 ASC NULLS FIRST], true
+- Aggregate [PULocationID#90, date_trunc(hour, lpep_pickup_datetime#86, Some(Asia/Singapore))], [PULocationID#90 AS revenue_zone#105, date_trunc(hour, lpep_pickup_datetime#86, Some(Asia/Singapore)) AS hour#106, sum(total_amount#101) AS amount#107, count(1) AS number_records#108L]
   +- Filter (lpep_pickup_datetime#86 >= cast(2020-01-01 00:00:00 as timestamp))
      +- SubqueryAlias green
         +- View (`green`, [VendorID#85, lpep_pickup_datetime#86, 

In [21]:
df_yellow = spark.read.option("recursiveFileLookup", "true").parquet('./data/pq/yellow/')

df_yellow.createOrReplaceTempView('yellow')


In [22]:
df_yellow_revenue = spark.sql("""
SELECT 
    -- Reveneue grouping 
    PULocationID AS revenue_zone,
    date_trunc('hour', tpep_pickup_datetime) AS hour, 
    
    SUM(total_amount) AS amount,
    count(1) as number_records
FROM
    yellow
where
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [36]:
df_yellow_revenue.repartition(20).write.parquet('./data/report/revenue/yellow', mode='overwrite')

25/07/25 18:02:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [29]:
df_green_revenue_tmp = df_green_revenue.withColumnRenamed('amount', 'green_amount')\
                        .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue.withColumnRenamed('amount', 'yellow_amount')\
                        .withColumnRenamed('number_records', 'yellow_number_records')

In [30]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['revenue_zone', 'hour'], how='outer')

In [57]:
df_join.repartition(20).write.parquet('./data/report/revenue/total', mode='overwrite')

25/07/25 18:18:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/07/25 18:18:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [70]:
df_join = spark.read.parquet('./data/report/revenue/total')

In [71]:
df_join.show()

+------------+-------------------+------------------+--------------------+------------------+---------------------+
|revenue_zone|               hour|      green_amount|green_number_records|     yellow_amount|yellow_number_records|
+------------+-------------------+------------------+--------------------+------------------+---------------------+
|          70|2020-08-11 21:00:00|              NULL|                NULL|118.49000000000001|                    3|
|          74|2020-01-15 01:00:00|              37.5|                   5|              85.6|                    8|
|         158|2020-12-10 14:00:00|              NULL|                NULL|249.46000000000006|                   16|
|         218|2021-05-12 15:00:00|             21.63|                   1|              NULL|                 NULL|
|          79|2021-04-08 01:00:00|              NULL|                NULL|            139.41|                   10|
|         148|2021-01-17 14:00:00|              NULL|                NUL

In [43]:
df_zones = spark.read.parquet('./zones')

In [44]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [72]:
df_result = df_join.join(df_zones, df_join.revenue_zone == df_zones.LocationID)

In [75]:
df_result.drop('LocationID', 'Zone').write.parquet('tmp/revenue-zones')

In [76]:
df_result.explain(True)

== Parsed Logical Plan ==
Join Inner, (cast(revenue_zone#383 as bigint) = cast(LocationID#309 as bigint))
:- Relation [revenue_zone#383,hour#384,green_amount#385,green_number_records#386L,yellow_amount#387,yellow_number_records#388L] parquet
+- Relation [LocationID#309,Borough#310,Zone#311,service_zone#312] parquet

== Analyzed Logical Plan ==
revenue_zone: int, hour: timestamp, green_amount: double, green_number_records: bigint, yellow_amount: double, yellow_number_records: bigint, LocationID: string, Borough: string, Zone: string, service_zone: string
Join Inner, (cast(revenue_zone#383 as bigint) = cast(LocationID#309 as bigint))
:- Relation [revenue_zone#383,hour#384,green_amount#385,green_number_records#386L,yellow_amount#387,yellow_number_records#388L] parquet
+- Relation [LocationID#309,Borough#310,Zone#311,service_zone#312] parquet

== Optimized Logical Plan ==
Join Inner, (cast(revenue_zone#383 as bigint) = cast(LocationID#309 as bigint))
:- Filter isnotnull(revenue_zone#383)
:

In [78]:
df_result.rdd.getNumPartitions()

7

In [80]:
spark.conf.set('spark.sql.shuffle.partitions', 50)

### Partitions test

In [81]:
df_join_test = spark.read.parquet('./data/report/revenue/total')

In [82]:
df_join_test.columns

['revenue_zone',
 'hour',
 'green_amount',
 'green_number_records',
 'yellow_amount',
 'yellow_number_records']

In [84]:
from pyspark.sql.functions import *
df_gb_test = df_join_test.groupBy('revenue_zone').agg(count('*'))

In [88]:
df_join.rdd.getNumPartitions()

7

In [87]:
df_gb_test.rdd.getNumPartitions()

50